In [1]:
from selenium.webdriver.common.keys import Keys 
from selenium.webdriver.chrome.options import Options

# pip install webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from datetime import datetime
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from time import sleep
from selenium.webdriver.common.keys import Keys


import pickle
import time 
import json
import re
import numpy as np
import urllib.request
import os
import json
import argparse
import shutil
import pandas as pd
import glob
import random


In [2]:
s = Service(ChromeDriverManager().install())

options = Options()
options.add_argument("--start-maximized")   # optional

driver = webdriver.Chrome(service=s, options=options)
wait = WebDriverWait(driver,15)


# First

In [3]:
wait = WebDriverWait(driver,15)

In [4]:
driver.get("https://afaq-stores.com/products")

In [5]:
num_of_pages = 551
pages_links = [
    f"https://afaq-stores.com/products?page={page}"
    for page in range(1, num_of_pages + 1)
        
]
len(pages_links)

551

In [6]:
def wait_for_page_load(timeout=10):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

In [7]:
def scroll_slowly_to_bottom(step=300, pause_time=0.5):
    """
    Scrolls slowly to the bottom of the page.
    
    step: how many pixels to scroll per iteration
    pause_time: seconds to wait between steps
    """
    last_height = driver.execute_script("return document.body.scrollHeight")
    current_position = 0

    while current_position < last_height:
        current_position += step
        driver.execute_script(f"window.scrollTo(0, {current_position});")
        time.sleep(pause_time)
        last_height = driver.execute_script("return document.body.scrollHeight")

In [8]:
products = []

def product_links(num_of_pages: int):
    for page in range(num_of_pages):

        driver.get(pages_links[page])
        wait_for_page_load(timeout=15)
        
        # Scroll slowly to the bottom to load all products
        scroll_slowly_to_bottom(step=300, pause_time=0.5)

        attributes = wait.until(EC.visibility_of_any_elements_located((By.CSS_SELECTOR, "div.product_text a")))
        for attrib in attributes:
            products.append(attrib.get_attribute("href"))
        print(len(products))

         # Dynamic wait using JavaScript to avoid sleep
        driver.execute_script(f"return new Promise(resolve => setTimeout(resolve, {random.randint(500, 1500)}));")

        # Save products list
        with open(r"C:\Users\nice\Desktop\Puresoft\Pures_Soft-Recommendation-System\Data\products\products_link_1.json", "w", encoding="utf-8") as f:
            json.dump(products, f, ensure_ascii=False, indent=4)

In [9]:
product_links(num_of_pages)

12
24
36
48
60
72
84
96
108
120
132
144
156
168
180
192
204
216
228
240
252
264
276
288
300
312
324
336
348
360
372
384
396
408
420
432
444
456
468
480
492
504
516
528
540
552
564
576
588
600
612
624
636
648
660
672
684
696
708
720
732
744
756
768
780
792
804


ReadTimeoutError: HTTPConnectionPool(host='localhost', port=58966): Read timed out. (read timeout=120)

In [25]:
len(list(set(products)))

6424

In [ ]:

# Save products list
with open(r"C:\Users\nice\Desktop\Puresoft\Pures_Soft-Recommendation-System\Data\products\products_link_1.json", "w", encoding="utf-8") as f:
    json.dump(products, f, ensure_ascii=False, indent=4)



In [ ]:
with open(r"C:\Users\nice\Desktop\Puresoft\Pures_Soft-Recommendation-System\Data\products\products_link_1.json", "r", encoding="utf-8") as f:
    products = json.load(f)

print(f"Loaded {len(products)} products")


Loaded 6612 products


In [26]:
len(products)

6612

In [23]:
def wait_for_page_load(timeout=10):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

In [ ]:
product_details = []


def get_product_details(products, length):
    for i in range(length):

        driver.get(products[i])
        wait_for_page_load(timeout=15)

        raw_name = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "details_title"))).text
        name = raw_name if raw_name else "None"
        
        raw_text_id = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "details_tags_sku"))).text
        match = re.search(r'\d+', raw_text_id) if raw_text_id else "None"
        product_id = match.group() if match else "None"

        raw_category = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "category"))).text
        category = raw_category if raw_category else "None"

        raw_stock = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "stock"))).text
        stock = raw_stock if raw_stock else "None"

        raw_text_price = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "price"))).text
        match = re.search(r'\d+', raw_text_price) if raw_text_price else None
        price = match.group() if match else "None"

        product_link = products[i]

        # Dynamic wait using JavaScript to avoid sleep
        driver.execute_script(f"return new Promise(resolve => setTimeout(resolve, {random.randint(500, 1000)}));")


        product_details.append( {
            "content" : name,
            "metadata": {
                "product_id": product_id,
                "category": category,
                "stock": stock,
                "price": price,
                "product_link": product_link
            }
        })
        if len(product_details) % 10 == 0:
            print(len(product_details))
            with open(r"C:\Users\nice\Desktop\Puresoft\Pures_Soft-Recommendation-System\Data\products\product_details_1.json", "w", encoding="utf-8") as f:
                json.dump(product_details, f, ensure_ascii=False, indent=4)




In [24]:
from selenium.common.exceptions import TimeoutException

def safe_get_text(by, value, wait, default="", timeout=3):
    try:
        return WebDriverWait(wait._driver, timeout).until(
            EC.presence_of_element_located((by, value))
        ).text.strip()
    except TimeoutException:
        return default

In [ ]:
#product_details = []


def get_product_details(products, length):
    for i in range(length):

        driver.get(products[i])
        wait_for_page_load(timeout=15)

        raw_name = safe_get_text(By.CLASS_NAME, "details_title", wait)
        name = raw_name if raw_name else ""
        
        raw_text_id = safe_get_text(By.CLASS_NAME, "details_tags_sku", wait)
        match = re.search(r'\d+', raw_text_id) if raw_text_id else ""
        product_id = match.group() if match else ""

        raw_category = safe_get_text(By.CLASS_NAME, "category", wait)
        category = raw_category if raw_category else ""

        raw_stock = safe_get_text(By.CLASS_NAME, "stock", wait)
        stock = raw_stock if raw_stock else ""

        raw_text_price = safe_get_text(By.CLASS_NAME, "price", wait)
        match = re.search(r'\d+', raw_text_price) if raw_text_price else None
        price = match.group() if match else ""

        product_link = products[i]

        # Dynamic wait using JavaScript to avoid sleep
        driver.execute_script(f"return new Promise(resolve => setTimeout(resolve, {random.randint(500, 1000)}));")


        product_details.append( {
            "content" : name,
            "metadata": {
                "product_id": product_id,
                "category": category,
                "stock": stock,
                "price": price,
                "product_link": product_link
            }
        })
       
        with open("product_details4.json", "w", encoding="utf-8") as f:
            json.dump(product_details, f, ensure_ascii=False, indent=4)




In [37]:
product_details = product_details[:4300]
len(product_details)

4300

In [38]:
len(products)

6612

In [39]:
get_product_details(products=products[4301:], length=len(products) - 4300)

4310
4320
4330
4340
4350
4360
4370
4380
4390
4400
4410
4420
4430
4440
4450
4460
4470
4480
4490
4500
4510
4520
4530
4540
4550
4560
4570
4580
4590
4600
4610
4620
4630
4640
4650
4660
4670
4680
4690
4700
4710
4720
4730
4740
4750
4760
4770
4780
4790
4800
4810
4820
4830
4840
4850
4860
4870
4880
4890
4900
4910
4920
4930
4940
4950
4960
4970
4980
4990
5000
5010
5020
5030
5040
5050
5060
5070
5080
5090
5100
5110
5120
5130
5140
5150
5160
5170
5180
5190
5200
5210
5220
5230
5240
5250
5260
5270
5280
5290
5300
5310
5320
5330
5340
5350
5360
5370
5380
5390
5400
5410
5420
5430
5440
5450
5460
5470
5480
5490
5500
5510
5520
5530
5540
5550
5560
5570
5580
5590
5600
5610
5620
5630
5640
5650
5660
5670
5680
5690
5700
5710
5720
5730
5740
5750
5760
5770
5780
5790
5800
5810
5820
5830
5840
5850
5860
5870
5880
5890
5900
5910
5920
5930
5940
5950
5960
5970
5980
5990
6000
6010
6020
6030
6040
6050
6060
6070
6080
6090
6100
6110
6120
6130
6140
6150
6160
6170
6180
6190
6200
6210
6220
6230
6240
6250
6260
6270
6280
6290
6300


IndexError: list index out of range

In [35]:
product_details[0]["metadata"]["product_link"]


'https://afaq-stores.com/product-details/9575'

In [38]:
links = []
for i in range(len(product_details)):
    links.append(product_details[i]["metadata"]["product_link"])

In [40]:
len(list(set(links)))

5867

In [40]:
len(product_details)

6611

In [41]:
product_details[0]

{'content': 'ACM ديبى وايت اى كونتور جل 15 مل @',
 'metadata': {'product_id': '3760095250205',
  'category': 'اى كونتور',
  'stock': 'متوفر في المخزون',
  'price': '370',
  'product_link': 'https://afaq-stores.com/product-details/11465'}}

In [42]:
def detect_and_remove_duplicates(data, key_path):
    """
    data: list of dicts like your sample
    key_path: list of nested keys, e.g. ['metadata', 'product_link']

    returns:
        unique_items: list of dicts
        duplicates: list of duplicated dicts
    """
    seen = set()
    unique_items = []
    duplicates = []

    for item in data:
        # get nested value
        value = item
        for k in key_path:
            value = value.get(k)
            if value is None:
                break

        if value in seen:
            duplicates.append(item)
        else:
            seen.add(value)
            unique_items.append(item)

    return unique_items, duplicates


In [43]:
unique_products, duplicated_products = detect_and_remove_duplicates(
    product_details, key_path=['metadata', 'product_link']
)

print("Total products:", len(product_details))
print("Unique products:", len(unique_products))
print("Duplicates:", len(duplicated_products))


Total products: 6611
Unique products: 6611
Duplicates: 0


In [44]:
# Save products list
with open("unique_products2.json", "w", encoding="utf-8") as f:
    json.dump(unique_products, f, ensure_ascii=False, indent=4)


In [45]:
len(unique_products)

6611

# Second

In [3]:
wait = WebDriverWait(driver,15)

In [4]:
driver.get("https://afaq-stores.com/products")

# new

In [3]:
session = {
    "user_id": "u123",
    "events": []
}

In [4]:
def is_product_page(url):
    return "product-details" in url

In [5]:
def log_event(event_type, link, duration=None):
    event = {
        "event_type": event_type,
        "timestamp": datetime.utcnow().isoformat(),
        "product_link": link
    }

    if duration is not None:
        event["duration"] = duration

    session["events"].append(event)

    with open("user_events2.json", "w") as f:
        json.dump(session, f, indent=2)

    print(session)


In [6]:
def inject_buy_tracker(driver):
    driver.execute_script("""
        window.buyClicked = false;

        const attach = () => {
            const btn = document.querySelector("a.common_btn.buy_now");
            if (btn && !btn.dataset.tracked) {
                btn.dataset.tracked = "true";
                btn.addEventListener("click", () => {
                    window.buyClicked = true;
                });
            }
        };

        // Try immediately
        attach();

        // Also watch for DOM changes (SPA pages, reloads, etc)
        const obs = new MutationObserver(attach);
        obs.observe(document.body, { childList: true, subtree: true });
    """)


In [7]:
def inject_cart_tracker(driver):
    driver.execute_script("""
        window.cartClicked = false;

        const attach = () => {
            const btn = document.querySelector("a.common_btn[wire\\\\:click\\\\.prevent='addToCart']");
            if (btn && !btn.dataset.tracked) {
                btn.dataset.tracked = "true";
                btn.addEventListener("click", () => {
                    window.cartClicked = true;
                });
            }
        };

        attach();
        const obs = new MutationObserver(attach);
        obs.observe(document.body, { childList: true, subtree: true });
    """)


In [13]:
on_product_page = False
start_time = None
current_product = None

while True:
    try:
        current_url = driver.current_url
        is_product = is_product_page(current_url)

        # ENTER product page
        if is_product and not on_product_page:
            log_event("product_open", current_url)
            start_time = time.time()
            current_product = current_url
            on_product_page = True

            inject_buy_tracker(driver)
            inject_cart_tracker(driver)

        # EXIT product page
        elif not is_product and on_product_page:
            duration = round(time.time() - start_time, 2)
            log_event("product_exit", current_product, duration)
            on_product_page = False
            current_product = None

        # Inside product page → check buttons
        if on_product_page:
            buy_clicked = driver.execute_script("return window.buyClicked;")
            cart_clicked = driver.execute_script("return window.cartClicked;")

            if buy_clicked:
                log_event("buy_click", current_product)
                driver.execute_script("window.buyClicked = false;")

            if cart_clicked:
                log_event("add_to_cart", current_product)
                driver.execute_script("window.cartClicked = false;")

        time.sleep(0.05)

    except Exception as e:
        print("Error:", e)
        time.sleep(1)


C:\Users\Mourad\AppData\Local\Temp\ipykernel_2148\3039934445.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


{'user_id': 'u123', 'events': [{'event_type': 'product_open', 'timestamp': '2026-02-13T21:44:57.386646', 'product_link': 'https://afaq-stores.com/product-details/32727'}]}
{'user_id': 'u123', 'events': [{'event_type': 'product_open', 'timestamp': '2026-02-13T21:44:57.386646', 'product_link': 'https://afaq-stores.com/product-details/32727'}, {'event_type': 'buy_click', 'timestamp': '2026-02-13T21:45:01.793934', 'product_link': 'https://afaq-stores.com/product-details/32727'}]}
{'user_id': 'u123', 'events': [{'event_type': 'product_open', 'timestamp': '2026-02-13T21:44:57.386646', 'product_link': 'https://afaq-stores.com/product-details/32727'}, {'event_type': 'buy_click', 'timestamp': '2026-02-13T21:45:01.793934', 'product_link': 'https://afaq-stores.com/product-details/32727'}, {'event_type': 'product_exit', 'timestamp': '2026-02-13T21:45:02.677923', 'product_link': 'https://afaq-stores.com/product-details/32727', 'duration': 5.29}]}
{'user_id': 'u123', 'events': [{'event_type': 'produ

KeyboardInterrupt: 